In [1]:
import pandas as pd

In [2]:
import sys
sys.path.append("../src")

In [3]:
from sqlalchemy import text
from db import crear_engine

engine = crear_engine()

with engine.connect() as conn:
    resultado = conn.execute(text("SELECT 1")).scalar()

print(resultado)

1


In [4]:
from cargar_mart import cargar_mart

df_mart = cargar_mart()
df_mart.shape

(3008016, 16)

In [5]:
from filtrar_entrenamiento import filtrar_entrenamiento

df_train = filtrar_entrenamiento(df_mart)

In [6]:
assert len(df_train) == 1609 * 1651
assert df_train.shape[1] == 16
assert df_mart.shape[1] == 16

## Ceros


In [7]:
df_mart['es_cero'] = df_mart['sales'] == 0
ceros_por_serie = df_mart.groupby(['store_number','family'])['es_cero'].transform('mean')

In [8]:
(ceros_por_serie <= 0.9).sum()

np.int64(2715992)

In [9]:
primera_fecha = df_mart['date'].min()
corte = primera_fecha + pd.Timedelta(days=37)
corte

Timestamp('2013-02-07 00:00:00')

In [10]:
mask = ((ceros_por_serie <= 0.9) & (df_mart['date'] >= corte))

In [11]:
assert len(df_mart[mask]) == 1609 * 1651